# Chapter 6 Exercise — Coffee Maker Recursive Decomposition

Decompose `BrewUnit` into component parts and record two judgment sites the decomposition raises, then a stopping judgment.

## Problem

Your model from Chapter 5 allocates `brew.applyWater` — the nested step
inside `CoffeeMaker`'s own inherited `brew` action — to `brewUnit`, the
`BrewUnit` usage `CoffeeMaker` composes: a usage-to-usage allocation, not
`BrewUnit` allocated to the `Brew` action definition. Work through these
steps:

1. Add a `throughput : ISQ::MassFlowRateValue default = 0.005 [SI::'kg⋅s⁻¹']`
   attribute to `BrewUnit`. Add a `BrewReq` requirement def with
   `subject bu : BrewUnit; require constraint { bu.throughput >= 0.003
   ['kg⋅s⁻¹'] }` and a `requirement brewReq : BrewReq;` usage. Create an
   `underSpec : BrewUnit` variant with `throughput` overridden to
   `0.001 ['kg⋅s⁻¹']`, folding `assert not satisfy brewReq by underSpec;`
   into its own body — mirroring `weak`'s own pattern for
   `HeatGenerationReq` in main Chapter 6 notebook 02.
2. Decompose `BrewUnit` into an `abstract part def BrewComponent`, a
   `part def Impeller :> BrewComponent`, and a `part def GroundsBasket :>
   BrewComponent` (not `FilterBasket`: Chapter 5's own exercise already
   declares a `FilterBasket` for the water-line filter, a real, distinct
   water-line component — reusing the name would collide). Create
   `part def BrewAssembly :> BrewUnit` that composes `part impeller :
   Impeller` and `part groundsBasket : GroundsBasket`.
3. Write an `asserted_context` record `AC-C06-EX` framing what kind of
   measure `brewReq` constrains: a measure of effectiveness or a measure
   of performance — mirroring `AC-C06`'s own framing of `heatGenerationReq`
   in main Chapter 6 notebook 02. Decide for yourself, and justify it: is
   the throughput threshold a component-level engineering rating, or a
   user-facing acceptance criterion?
4. Write an `asserted_solution` record `AS-C06-EX`: which mechanism
   `Impeller` represents, and which plausible alternative it is selected
   over — mirroring `AS-C06`'s own mechanism-selection reasoning (a
   confirmed model fact plus a domain premise about how the two
   mechanisms respond to a discrete control signal, not a claim the model
   already connects one and not the other).
5. Write an `asserted_inference` record `AI-C06-EX`, honestly scoped
   exactly like `AI-C06` and `AI-C04-EX`: state plainly what this branch's
   structural decomposition establishes and what it does not — **not** a
   claim that the `BrewUnit` decomposition is complete. Set `premises` to
   `["AC-C06-EX", "AS-C06-EX", "AS-C03-EX", "AI-C04-EX"]`, the
   coffee-domain analogues of `AI-C06`'s own four premises.

Verify with `model.ok == True` and `validate_record(...) == []` for all
three records.

In [ ]:
import opensysml
from toaster.report import format_diagnostics

conn = opensysml.connect(version="v0.9.0")

# Paste your completed Ch5 model here, then add the throughput attribute,
# the BrewReq requirement, and the underSpec fixture.
source = """
package CoffeeDemo {
    private import ScalarValues::*;
    private import SI::*;
    private import ISQ::*;

    item def Beans;
    item def Coffee;

    item def BrewStart {
        doc /* Signal marking the start of a brewing cycle, not the beans or coffee material flow. */
    }
    item def BrewFinish {
        doc /* Signal marking the finish of a brewing cycle, not the beans or coffee material flow. */
    }
    item def BrewCancel {
        doc /* Signal requesting cancellation of an in-progress brewing cycle, not the beans or coffee material flow. */
    }

    action def ApplyWater {
        in beans : Beans;
        in water : ISQ::MassValue[0..*];
        in duration : ISQ::DurationValue[0..*];
        out coffee : Coffee;
        out delivered : ISQ::MassValue;
        out retained : ISQ::MassValue;
        assert constraint balance {
            delivered >= 0.0 [SI::kg] and retained >= 0.0 [SI::kg] and delivered + retained <= water
        }
    }

    action def Brew {
        doc /* Transform beans into coffee acceptable to its user. */
        in beans : Beans;
        out coffee : Coffee;
        first start;
        then action applyWater : ApplyWater {
            in beans = Brew::beans;
        }
        then done;
    }

    abstract part def BrewingSystem {
        perform action brew : Brew;
    }

    part def BrewUnit {
        attribute brewTemp : ISQ::ThermodynamicTemperatureValue;
        attribute throughput : ISQ::MassFlowRateValue default = 0.005 ['kg⋅s⁻¹'];
    }
    part def HeatExchanger;

    part def CoffeeMaker :> BrewingSystem {
        part brewUnit : BrewUnit;
        part heatExchanger : HeatExchanger;
        allocation brewAllocation allocate brew.applyWater to brewUnit;
    }

    requirement def TemperatureReq {
        doc /*
         * The brew unit shall operate at at most 96 degrees Celsius (369.15 K)
         * to avoid over-extracting the grounds.
         */
        subject bu : BrewUnit;
        require constraint { bu.brewTemp <= 369.15 [SI::K] }
    }

    requirement tempCheck : TemperatureReq;

    part nominal : BrewUnit;
    part hot : BrewUnit {
        attribute :>> brewTemp = 371.15 [SI::K];
        assert not satisfy tempCheck by hot;
    }

    verification def BrewTempTest {
        subject bu : BrewUnit;
        objective {
            verify tempCheck;
        }
    }

    port def WaterPort {
        out water : ISQ::MassValue[0..*];
    }

    part def WaterPump {
        port waterOut : WaterPort;
    }
    part def FilterBasket {
        port waterIn : ~WaterPort;
    }
    part def CoffeeFlow {
        part pump : WaterPump;
        part filterUnit : FilterBasket;
        interface waterInterface connect pump.waterOut to filterUnit.waterIn;
    }

    requirement def BrewReq {
        doc /*
         * The brew unit shall move water through the grounds at at least
         * 0.003 kg/s. This is an engineering performance threshold on a
         * component rating, not yet derived from a stated measure of
         * effectiveness: no stated brew-time or taste acceptance criterion
         * connects to it yet. Recorded openly, not faked.
         */
        subject bu : BrewUnit;
        require constraint { bu.throughput >= 0.003 ['kg⋅s⁻¹'] }
    }

    requirement brewReq : BrewReq;

    part underSpec : BrewUnit {
        attribute :>> throughput = 0.001 ['kg⋅s⁻¹'];
        assert not satisfy brewReq by underSpec;
    }
}
"""

model = conn.load_from_content(source, strict=False)
print(f"Model ok: {model.ok}")
if not model.ok:
    print(format_diagnostics(model.diagnostics))

In [ ]:
# Step 2: add the second-level structural decomposition.
source_with_decomp = """
package CoffeeDemo {
    private import ScalarValues::*;
    private import SI::*;
    private import ISQ::*;

    item def Beans;
    item def Coffee;

    item def BrewStart {
        doc /* Signal marking the start of a brewing cycle, not the beans or coffee material flow. */
    }
    item def BrewFinish {
        doc /* Signal marking the finish of a brewing cycle, not the beans or coffee material flow. */
    }
    item def BrewCancel {
        doc /* Signal requesting cancellation of an in-progress brewing cycle, not the beans or coffee material flow. */
    }

    action def ApplyWater {
        in beans : Beans;
        in water : ISQ::MassValue[0..*];
        in duration : ISQ::DurationValue[0..*];
        out coffee : Coffee;
        out delivered : ISQ::MassValue;
        out retained : ISQ::MassValue;
        assert constraint balance {
            delivered >= 0.0 [SI::kg] and retained >= 0.0 [SI::kg] and delivered + retained <= water
        }
    }

    action def Brew {
        doc /* Transform beans into coffee acceptable to its user. */
        in beans : Beans;
        out coffee : Coffee;
        first start;
        then action applyWater : ApplyWater {
            in beans = Brew::beans;
        }
        then done;
    }

    abstract part def BrewingSystem {
        perform action brew : Brew;
    }

    part def BrewUnit {
        attribute brewTemp : ISQ::ThermodynamicTemperatureValue;
        attribute throughput : ISQ::MassFlowRateValue default = 0.005 ['kg⋅s⁻¹'];
    }
    part def HeatExchanger;

    part def CoffeeMaker :> BrewingSystem {
        part brewUnit : BrewUnit;
        part heatExchanger : HeatExchanger;
        allocation brewAllocation allocate brew.applyWater to brewUnit;
    }

    requirement def TemperatureReq {
        doc /*
         * The brew unit shall operate at at most 96 degrees Celsius (369.15 K)
         * to avoid over-extracting the grounds.
         */
        subject bu : BrewUnit;
        require constraint { bu.brewTemp <= 369.15 [SI::K] }
    }

    requirement tempCheck : TemperatureReq;

    part nominal : BrewUnit;
    part hot : BrewUnit {
        attribute :>> brewTemp = 371.15 [SI::K];
        assert not satisfy tempCheck by hot;
    }

    verification def BrewTempTest {
        subject bu : BrewUnit;
        objective {
            verify tempCheck;
        }
    }

    port def WaterPort {
        out water : ISQ::MassValue[0..*];
    }

    part def WaterPump {
        port waterOut : WaterPort;
    }
    part def FilterBasket {
        port waterIn : ~WaterPort;
    }
    part def CoffeeFlow {
        part pump : WaterPump;
        part filterUnit : FilterBasket;
        interface waterInterface connect pump.waterOut to filterUnit.waterIn;
    }

    requirement def BrewReq {
        doc /*
         * The brew unit shall move water through the grounds at at least
         * 0.003 kg/s. This is an engineering performance threshold on a
         * component rating, not yet derived from a stated measure of
         * effectiveness: no stated brew-time or taste acceptance criterion
         * connects to it yet. Recorded openly, not faked.
         */
        subject bu : BrewUnit;
        require constraint { bu.throughput >= 0.003 ['kg⋅s⁻¹'] }
    }

    requirement brewReq : BrewReq;

    part underSpec : BrewUnit {
        attribute :>> throughput = 0.001 ['kg⋅s⁻¹'];
        assert not satisfy brewReq by underSpec;
    }

    abstract part def BrewComponent;

    part def Impeller :> BrewComponent {
        doc /* A motor-driven centrifugal impeller: moves water through the
             * grounds by direct rotation. The mechanism selection this
             * specialization commits to is recorded (AS-C06-EX) against the
             * alternative it was chosen over. */
    }
    part def GroundsBasket :> BrewComponent {
        doc /* Holds the coffee grounds the moved water passes through. Named
             * GroundsBasket, not FilterBasket: Chapter 5's own exercise
             * already declares a FilterBasket for the water-line filter, a
             * real, distinct water-line component. */
    }

    part def BrewAssembly :> BrewUnit {
        part impeller : Impeller;
        part groundsBasket : GroundsBasket;
    }
}
"""

model2 = conn.load_from_content(source_with_decomp, strict=False)
print(f"Model with decomp ok: {model2.ok}")
if not model2.ok:
    print(format_diagnostics(model2.diagnostics))

part_defs = [e.as_dict() for e in model2.query()
             if e.as_dict().get("@type") == "PartDefinition"]
print(f"Part definitions: {[p.get('declaredName') or p.get('name') for p in part_defs]}")

In [ ]:
# Step 3: write the asserted_context framing record.
from toaster.evidence import ReviewRecord, hash_content, validate_record

framing_claim = (
    "brewReq (BrewReq) is framed as a measure of performance: an engineering "
    "rating on BrewUnit's own throughput, not a direct measure of the user's "
    "acceptance of the coffee."
)
framing_record = ReviewRecord(
    identifier="AC-C06-EX",
    kind="asserted_context",
    claim=framing_claim,
    model_ref="CoffeeDemo::brewReq",
    content_hash=hash_content(source_with_decomp),
    scope="CoffeeDemo",
    criteria=(
        "MoE if the split names who cares and frames the measure as "
        "acceptance; MoP if its threshold is derived from a stated MoE with "
        "a means of checking (architecture-layers skill)."
    ),
    premises=[],
    assumption_refs=[
        "The MoE/MoP split for a component rating is a case-specific "
        "modeling judgment, not a fixed rule (AC-C06 makes the same split "
        "for HeatGenerationReq; AC-C03-EX makes the opposite split for "
        "TemperatureReq's own brew-temperature threshold, which names a "
        "stated over-extraction acceptance criterion directly)."
    ],
    evidence_refs=[
        "CoffeeDemo::BrewReq doc: the rationale states a component "
        "throughput rating threshold, naming no stakeholder acceptance "
        "criterion (contrast TemperatureReq's own rationale, which names "
        "over-extraction directly)."
    ],
    rationale=(
        "BrewReq's rationale argues from BrewUnit's own throughput rating, "
        "not from what a user tastes or notices about the coffee: it names "
        "no one who would reject a coffee maker on this figure alone, and a "
        "brew unit rated below 0.003 kg/s could still make acceptable "
        "coffee more slowly. Framing it as a MoP is honest about who cares "
        "(the engineer sizing the impeller) and what the threshold "
        "characterizes (a flow-rate rating), not the user's acceptance -- "
        "the same shape AC-C06 uses for HeatGenerationReq's own power "
        "rating."
    ),
    counterevidence=(
        "A user might notice a brew unit so slow that brewing takes too "
        "long, or so fast that water passes through the grounds too "
        "quickly to extract properly (under-extraction), which links this "
        "rating back to taste and brew time indirectly. The link is not "
        "modeled: no relation connects throughput, water volume and brew "
        "time yet, so treating throughput as purely engineering-internal "
        "is a simplification, the same open point AC-C06 records for "
        "power and toast time."
    ),
    residual_uncertainties=(
        "The 0.003 kg/s threshold is not derived from any stated measure "
        "of effectiveness (a brew-time or taste acceptance criterion) "
        "through a modeled relation: no water-volume or brew-time relation "
        "exists together with throughput in this model to derive it from. "
        "It is recorded as a free-standing engineering figure, honestly, "
        "not a fixed rule for every future chapter."
    ),
    disposition="pending",
    dependency_freshness="current",
    engineering_conclusion="undetermined",
    record_kind="worked_example",
)

framing_errors = validate_record(framing_record)
print(f"AC-C06-EX validation errors: {framing_errors}")

In [ ]:
# Step 4: write the asserted_solution mechanism-selection record.
brew_start = model2.find("CoffeeDemo::BrewStart")
brew_cancel = model2.find("CoffeeDemo::BrewCancel")
print(f"BrewStart: {brew_start.kind}, BrewCancel: {brew_cancel.kind}")

selection_claim = (
    "Impeller, a motor-driven centrifugal mechanism that moves water "
    "through the grounds by direct rotation, is selected over a "
    "gravity-fed drip mechanism (no motor, no direct actuation point) as "
    "the mechanism BrewComponent commits to."
)
selection_record = ReviewRecord(
    identifier="AS-C06-EX",
    kind="asserted_solution",
    claim=selection_claim,
    model_ref="CoffeeDemo::Impeller",
    content_hash=hash_content(source_with_decomp),
    scope="CoffeeDemo::BrewComponent and its realizations",
    criteria=(
        "The chosen mechanism must respond to a discrete start/stop "
        "signal like BrewStart/BrewCancel (Chapter 4), and must expose a "
        "rating brewReq's throughput threshold can be checked against "
        "once a concrete part exists."
    ),
    premises=[
        "CoffeeDemo declares BrewStart and BrewCancel (Chapter 4), "
        "discrete signals marking the start and cancellation of a brewing "
        "cycle, confirmed by model.find() above.",
        "Domain premise, not derived from the model: a motor-driven "
        "impeller responds to being switched on and off directly, while a "
        "gravity-fed drip mechanism has no actuation point of its own and "
        "needs a separate valve or timing mechanism to start or stop the "
        "flow on command. Neither BrewComponent nor any of its "
        "realizations is yet connected to a control signal in this model; "
        "this premise is about the physical mechanisms themselves, not "
        "about what the model currently wires together.",
    ],
    assumption_refs=[
        "BrewComponent commits to no water-moving mechanism (it is a bare "
        "abstract base): this record is what actually commits to a "
        "motor-driven form, not a fact already built into BrewComponent "
        "itself."
    ],
    evidence_refs=[
        f"model.find('CoffeeDemo::BrewStart') resolves to a real "
        f"{brew_start.kind}, confirmed above.",
        f"model.find('CoffeeDemo::BrewCancel') resolves to a real "
        f"{brew_cancel.kind}, confirmed above.",
    ],
    rationale=(
        "A motor-driven impeller responds to being switched on and off "
        "directly, the same shape as a discrete start/cancel signal, "
        "while a gravity-fed drip mechanism needs a separate valve or "
        "timing device to respond the same way (the domain premise "
        "above). That is a claim about how the two mechanisms work, not "
        "something this model currently shows: no realization of "
        "BrewComponent is yet connected to any control signal, so this "
        "selection is a reasoned engineering preference argued from "
        "mechanism, not a claim that the model already connects one "
        "mechanism and not the other."
    ),
    counterevidence=(
        "This does not rule out a gravity-fed design: a drip mechanism "
        "controlled by its own timed valve could equally respond to a "
        "discrete signal, which is exactly why the argument above rests "
        "on a domain premise about how the two mechanisms work, not on "
        "anything the model itself already builds or connects. Neither "
        "mechanism's own flow-rate relation (impeller speed to "
        "throughput, or drip valve opening to throughput) is modeled, so "
        "efficiency and response-time comparisons remain out of reach "
        "either way."
    ),
    residual_uncertainties=(
        "Once a control policy and a real actuation interface are "
        "modeled together, this selection could be revisited against a "
        "real trade study rather than a domain premise alone."
    ),
    disposition="pending",
    dependency_freshness="current",
    engineering_conclusion="undetermined",
    record_kind="worked_example",
)

selection_errors = validate_record(selection_record)
print(f"AS-C06-EX validation errors: {selection_errors}")

In [ ]:
# Step 5: write the stopping judgment, honestly scoped exactly like AI-C06
# and AI-C04-EX -- not a claim that the BrewUnit decomposition is complete.
nominal_holds = model2.eval("CoffeeDemo::brewReq(CoffeeDemo::nominal)")
underspec_holds = model2.eval("CoffeeDemo::brewReq(CoffeeDemo::underSpec)")
print(f"brewReq(nominal) = {nominal_holds}, brewReq(underSpec) = {underspec_holds}")

part_names = [p.get("declaredName") or p.get("name") for p in part_defs]

claim = (
    "BrewAssembly is a real structural decomposition of BrewUnit: it "
    "specializes BrewUnit and composes impeller (Impeller) and "
    "groundsBasket (GroundsBasket), confirmed by querying the loaded "
    "model's part definitions directly, not merely declared syntax. "
    "brewReq evaluates against two real candidates, nominal (True, at "
    "BrewUnit's own default throughput) and underSpec (False), a genuine "
    "satisfaction check on an underived threshold, not an unevaluated "
    "assertion."
)

inference_record = ReviewRecord(
    identifier="AI-C06-EX",
    kind="asserted_inference",
    claim=claim,
    model_ref="CoffeeDemo::BrewAssembly",
    content_hash=hash_content(source_with_decomp),
    scope="CoffeeDemo::BrewAssembly and its realizations",
    criteria=(
        "Per the recursion's own stopping rule, a leaf performs its "
        "specified behavior, connects through its specified interfaces, "
        "and has verification evidence. At this level: BrewAssembly is a "
        "real structural decomposition of BrewUnit into impeller and "
        "groundsBasket (met, as containment); neither subpart performs "
        "any function of its own and no allocation assigns any behavior "
        "to either one, unlike HeatGenerator's own "
        "GenerateHeat/heatGenAllocation chain one level down in the "
        "toaster (not met). No interface connects impeller and "
        "groundsBasket (not met). brewReq evaluates on BrewUnit-level "
        "candidates (nominal, underSpec) against a threshold that is "
        "itself not yet derived, and this check is on BrewUnit's own "
        "attribute, not on BrewAssembly or either subpart specifically "
        "(partial evidence, not full verification)."
    ),
    premises=["AC-C06-EX", "AS-C06-EX", "AS-C03-EX", "AI-C04-EX"],
    assumption_refs=[
        "BrewAssembly's own composition of impeller and groundsBasket is "
        "printed as part of source_with_decomp and loaded successfully in "
        "step 2; this does not by itself mean either subpart performs any "
        "function or is checked against any requirement of its own."
    ],
    evidence_refs=[
        f"model2.query() part definitions: {part_names}",
        f"brewReq(nominal) = {nominal_holds}, brewReq(underSpec) = "
        f"{underspec_holds}, evaluated directly against the loaded model "
        f"above.",
    ],
    rationale=(
        "Performs: not met. BrewAssembly's own composition of impeller "
        "and groundsBasket is real, confirmed directly by querying the "
        "loaded model's part definitions above, not just present in the "
        "source text -- but neither subpart performs any function of its "
        "own, and no allocation assigns any behavior to either one, "
        "unlike HeatGenerator performing GenerateHeat one level down in "
        "the toaster example. Connects: not met. No interface or port "
        "connects impeller and groundsBasket. Verified: partially met. "
        "brewReq is genuinely evaluated, not left as an unevaluated "
        "assertion, on two real candidates, one passing and one "
        "deliberately failing for a reason about the design -- but the "
        "check is on BrewUnit's own throughput attribute, not on "
        "BrewAssembly or either subpart individually, and its own "
        "threshold is not yet derived from any stated measure of "
        "effectiveness (AC-C06-EX), so this is partial verification "
        "evidence at best."
    ),
    counterevidence=(
        "impeller and groundsBasket have no function or allocation of "
        "their own nested at this level, unlike HeatGenerator's own real "
        "GenerateHeat/heatGenAllocation chain; no interface connects "
        "them; brewReq is checked only at BrewUnit's own level (nominal, "
        "underSpec), not against BrewAssembly or either subpart directly; "
        "BrewAssembly is not yet composed into any full CoffeeMaker "
        "candidate (CoffeeMaker::brewUnit is still typed by the abstract "
        "BrewUnit, not by BrewAssembly). This branch addresses only "
        "BrewUnit's own structural containment, not Brew's other flows "
        "(beans, coffee) or CoffeeMaker's other subsystem, HeatExchanger. "
        "This is a deliberate one-branch worked example, the same kind of "
        "honestly scoped choice main Chapter 6 made for GenerateHeat's "
        "own energy-to-heat conversion, not a claim that this exercise "
        "finishes BrewUnit's decomposition."
    ),
    residual_uncertainties=(
        "Whether Impeller or GroundsBasket need their own nested function "
        "and allocation, whether BrewAssembly is ever composed into a "
        "real CoffeeMaker candidate, and whether Brew's other flows get "
        "their own level-2 branches, are open for whichever exercise "
        "takes them up."
    ),
    disposition="pending",
    dependency_freshness="current",
    engineering_conclusion="supported",
    record_kind="worked_example",
)

errors = validate_record(inference_record)
print(f"AI-C06-EX validation errors: {errors}")
print(f"Premises: {inference_record.premises}")
conn.close()